# Student Name: Ghiyas Nizamudden Shaik

# Instructor: Yasser Abduallah

## CS634 End Term Project: Implementation of Supervised Data Mining for Binary Classification

### Import libraries

In [1]:
#import the necessery libraries

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

tf.random.set_seed(42)
pd.options.display.max_columns = None

### Import data

In [2]:
# import data

train = pd.read_csv("data/airline_train.csv", index_col=0)
train = train.set_index("id")

### Fill missing values

In [3]:
# fill the missing arrival delay with departure delay times assuming they will don't lose or gain time

train.loc[:, "Arrival Delay in Minutes"] = train.loc[:, "Arrival Delay in Minutes"].fillna(train.loc[:, "Departure Delay in Minutes"])

### Categorical Encoding

In [4]:
# encode the categorical columns into numerical values

cat_encod = {
    "Gender": ['Male', 'Female'],
    "Customer Type": {'Loyal Customer': 1, 'disloyal Customer': -1},
    "Type of Travel": ['Personal Travel', 'Business travel'],
    "Class": {'Eco Plus': 1, 'Business': 2, 'Eco': 0},
    "satisfaction": {'neutral or dissatisfied':0, 'satisfied': 1}
    
}

for col, val in cat_encod.items():
    if type(val) == dict:
        train.loc[:, col] = train.loc[:, col].map(val)
        train[col] = train[col].astype(int)
        
    else:
        train.loc[:, val] = pd.get_dummies(train.loc[:, col])
        train[val] = train[val].astype(int)
        train = train.drop(col, axis=1)

### Split features and target variables

In [5]:
# extract predictor and target variables

y = train["satisfaction"]
X = train.drop("satisfaction", axis=1)

### Data Transformation

In [6]:
# scale the data

scaler = StandardScaler()
X = scaler.fit_transform(X)

### Evaluating Classifiers

TP, TN, FP, FN, TPR(sensitivity, r), TNR(specificity)<br>
FPR, FNR, FDR, NPV, p, F1, acc, err<br>
BACC, TSS, HSS, BS, BSS<br>

In [7]:
# returns confusion matrix values given predicted and ground truth values

def confusion_matrix(preds, truth):
    tp = np.sum((preds == 1) & (truth == 1))
    fp = np.sum((preds == 1) & (truth == 0))
    tn = np.sum((preds == 0) & (truth == 0))
    fn = np.sum((preds == 0) & (truth == 1))
    return (tp, fp, tn, fn)

In [8]:
# returns a dictionary of different metrics given the confusion matrix

def metrics(tp, fp, tn, fn):
    model_metrics = {}
    tpr = tp / (tp + fn)
    model_metrics["tpr"] = tpr
    tnr = tn / (fp + tn)
    model_metrics["tnr"] = tnr
    fpr = fp / (fp + tn)
    model_metrics["fpr"] = fpr
    fnr = fn / (tp + fn)
    model_metrics["fnr"] = fnr
    
    model_metrics["recall"] = tpr
    prec = tp / (tp + fp)
    model_metrics["precision"] = prec
    f1 = 2 * prec * tpr / (prec + tpr)
    model_metrics["f1-score"] = f1
    acc = (tp + tn) / (tp + fn + fp + tn)
    model_metrics["accuracy"] = acc
    err = (fp + fn) / (tp + fn + fp + tn)
    model_metrics["error rate"] = err
    npv = tn / (tn + fn)
    model_metrics["npv"] = npv
    fdr = fp / (fp + tp)
    model_metrics["fdr"] = fdr
    
    bacc = (tpr + tnr) / 2
    model_metrics["bacc"] = bacc
    tss = tpr - fpr
    model_metrics["tss"] = tss
    hss = 2 * (tp * tn - fp * fn) / ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn))
    model_metrics["hss"] = hss

    return model_metrics

In [9]:
metric_list=["tpr", "tnr", "fpr", "fnr", "recall", "precision", "f1-score", "accuracy", "error rate", "npv", "fdr",
                                    "bacc", "tss", "hss"]

In [10]:
# use k-fold with 10 splits

kf = KFold(n_splits=10, shuffle=True, random_state=42)

### Random Forest

In [11]:
rf_metrics = pd.DataFrame([], columns=metric_list)

for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    rf.fit(X_train, y_train)
    pred_proba = rf.predict_proba(X_test)[:, 1]
    preds = (pred_proba >= 0.5).astype(int)

    tp, fp, tn, fn = confusion_matrix(preds, y_test)
    model_metrics = metrics(tp, fp, tn, fn)
    rf_metrics.loc[i+1,:] = model_metrics
    
rf_avg = rf_metrics.mean()

In [12]:
print(rf_metrics)

         tpr       tnr       fpr       fnr    recall precision  f1-score  \
1   0.942737  0.977846  0.022154  0.057263  0.942737  0.970414  0.956375   
2   0.939054  0.977078  0.022922  0.060946  0.939054  0.969559  0.954063   
3   0.941745  0.976036  0.023964  0.058255  0.941745  0.968354  0.954865   
4    0.94386  0.980449  0.019551   0.05614   0.94386  0.974196  0.958788   
5   0.937803  0.974044  0.025956  0.062197  0.937803  0.965486  0.951443   
6   0.939911  0.976871  0.023129  0.060089  0.939911  0.968914  0.954192   
7   0.946691  0.974677  0.025323  0.053309  0.946691  0.965223  0.955867   
8   0.943182  0.977126  0.022874  0.056818  0.943182  0.969093  0.955962   
9   0.941672  0.977725  0.022275  0.058328  0.941672  0.970071  0.955661   
10  0.948402  0.977537  0.022463  0.051598  0.948402  0.968524  0.958357   

    accuracy error rate       npv       fdr      bacc       tss       hss  
1   0.962564   0.037436  0.956812  0.029586  0.960292  0.920583  0.923602  
2   0.96044

### Deep Learning: LSTM

In [13]:
lstm_metrics = pd.DataFrame([], columns=metric_list)

for i, (train_index, test_index) in enumerate(kf.split(X_train)):
    model = Sequential([
        Input(shape=(X_train.shape[1], 1)),
        LSTM(50, return_sequences=False),
        Dense(1, activation='sigmoid')
    ])

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    pred_proba = model.predict(X_test)
    preds = (pred_proba >= 0.5).astype(int)

    tp, fp, tn, fn = confusion_matrix(preds.reshape(-1), y_test)
    model_metrics = metrics(tp, fp, tn, fn)
    lstm_metrics.loc[i+1,:] = model_metrics
    
lstm_avg = lstm_metrics.mean()

293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step   
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  


In [14]:
print(lstm_metrics)

         tpr       tnr       fpr       fnr    recall precision  f1-score  \
1   0.933595  0.966041  0.033959  0.066405  0.933595  0.955127  0.944238   
2   0.916525  0.976104  0.023896  0.083475  0.916525  0.967965  0.941543   
3   0.926647  0.967077  0.032923  0.073353  0.926647  0.957505  0.941824   
4   0.941597  0.954855  0.045145  0.058403  0.941597  0.941133  0.941365   
5   0.914222  0.971664  0.028336  0.085778  0.914222  0.960554  0.936815   
6   0.922869  0.967045  0.032955  0.077131  0.922869  0.955736  0.939015   
7   0.916646  0.972561  0.027439  0.083354  0.916646  0.963124  0.939311   
8   0.916667  0.975108  0.024892  0.083333  0.916667  0.965064  0.940243   
9   0.921564  0.970384  0.029616  0.078436  0.921564  0.959057  0.939937   
10  0.908348  0.976629  0.023371  0.091652  0.908348  0.965536  0.936069   

    accuracy error rate       npv       fdr      bacc       tss       hss  
1   0.951882   0.048118  0.949469  0.044873  0.949818  0.899635   0.90193  
2    0.9498

### Algorithms: KNN

In [15]:
knn_metrics = pd.DataFrame([], columns=metric_list)

for i, (train_index, test_index) in enumerate(kf.split(X_train)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    pred_proba = knn.predict_proba(X_test)[:, 1]
    preds = (pred_proba >= 0.5).astype(int)

    tp, fp, tn, fn = confusion_matrix(preds, y_test)
    model_metrics = metrics(tp, fp, tn, fn)
    knn_metrics.loc[i+1,:] = model_metrics

knn_avg = knn_metrics.mean()

In [16]:
print(knn_metrics)

         tpr       tnr       fpr       fnr    recall precision  f1-score  \
1   0.888735  0.959409  0.040591  0.111265  0.888735  0.942748  0.914945   
2   0.877739  0.961229  0.038771  0.122261  0.877739  0.946616  0.910877   
3   0.877434  0.967715  0.032285  0.122566  0.877434  0.954085  0.914155   
4   0.871519  0.960117  0.039883  0.128481  0.871519  0.943019   0.90586   
5   0.883936   0.96179   0.03821  0.116064  0.883936  0.947276  0.914511   
6   0.874197  0.965385  0.034615  0.125803  0.874197  0.952742  0.911781   
7   0.873255  0.962419  0.037581  0.126745  0.873255  0.946869  0.908573   
8   0.871916  0.965689  0.034311  0.128084  0.871916  0.950151  0.909354   
9   0.876359  0.963471  0.036529  0.123641  0.876359  0.949088  0.911274   
10  0.883727  0.962871  0.037129  0.116273  0.883727  0.946663  0.914113   

    accuracy error rate       npv       fdr      bacc       tss       hss  
1   0.929072   0.070928  0.919776  0.057252  0.924072  0.848144  0.854211  
2   0.92455

In [17]:
# average of all metrics for the three models

final_df = pd.concat([rf_avg, lstm_avg, knn_avg], axis=1)
final_df.columns=["Random Forest", "LSTM", "KNN"]
print(final_df)

           Random Forest      LSTM       KNN
tpr             0.942506  0.921868  0.877882
tnr             0.976939  0.969747  0.963009
fpr             0.023061  0.030253  0.036991
fnr             0.057494  0.078132  0.122118
recall          0.942506  0.921868  0.877882
precision       0.968984   0.95908  0.947926
f1-score        0.955557  0.940036  0.911544
accuracy        0.962003  0.949024  0.926048
error rate      0.037997  0.050976  0.073952
npv             0.956898  0.942016  0.911345
fdr             0.031016   0.04092  0.052074
bacc            0.959722  0.945807  0.920446
tss             0.919445  0.891615  0.840891
hss             0.922381  0.895716  0.848164
